# RAG-Anything V5 — Ingestion sur Kaggle GPU

Ce notebook ingère les documents PDF dans le knowledge graph LightRAG **sur GPU Kaggle (T4 16GB)**.

**Stack V5 (optimisée pour documents juridiques FR) :**
- Parser : **Docling** (meilleure extraction de tableaux que MinerU)
- LLM réponses : **`mistral-nemo:12b`** — meilleur modèle français à cette taille, anti-hallucination
- LLM extraction : **`qwen2.5:7b`** — rapide et précis pour l'extraction structurée d'entités
- Embeddings : **`nomic-embed-text`** — bon multilingue
- Chunking : 1200 tokens, **overlap 300** (vs 150) pour ne rien perdre aux frontières
- Retrieval : **top_k=25, chunk_top_k=15** — plus de contexte pour meilleur rappel
- Prompts : **citations obligatoires** du contexte → anti-hallucination

**Prérequis Kaggle :**
1. Activer le GPU : `Settings > Accelerator > GPU T4 x2` (ou T4 x1)
2. Activer Internet : `Settings > Internet > On`
3. Uploader vos PDFs comme Dataset Kaggle, ou les uploader directement

In [ ]:
# ── Cellule 1 : Installer Ollama sur Kaggle ──────────────────────────────────
!apt-get install -y zstd -q  # requis par l'installeur Ollama
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installé")

In [ ]:
# ── Cellule 2 : Démarrer Ollama + télécharger les modèles ───────────────────
import subprocess, time, os

# Démarrer le serveur Ollama en arrière-plan
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "2h"         # maintenir les modèles en VRAM 2h (défaut = 5min)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)
time.sleep(5)
print("✅ Ollama démarré")

# Modèles V5 optimisés pour documents juridiques français
models = [
    "mistral-nemo:12b",    # LLM réponses — meilleur FR, anti-hallucination (~7.5GB Q4)
    "qwen2.5:7b",          # LLM extraction entités — rapide, bon pour tâches structurées (~4.5GB Q4)
    "nomic-embed-text",    # Embeddings multilingues (~300MB)
]

for model in models:
    print(f"\n📥 Pull {model}...")
    !ollama pull {model}

print("\n✅ Tous les modèles sont prêts")
print("\nℹ️  llava:7b (vision) NON installé — pas nécessaire pour des PDFs texte.")
print("    Ajoutez '!ollama pull llava:7b' si vos PDFs contiennent des images.")

In [ ]:
# ── Cellule 3 : Cloner le projet + installer les dépendances ─────────────────
import os
os.chdir("/kaggle/working")
!rm -rf rag-anything-project

# Cloner votre repo (remplacer par votre URL si privé)
!git clone https://github.com/Abdelmajid-BARANI/rag-anything-project.git
os.chdir("/kaggle/working/rag-anything-project")

# Installer les dépendances (docling au lieu de mineru)
!pip install -r requirements.txt -q

print("✅ Projet installé avec Docling")

In [ ]:
# ── Cellule 4 : Préparer les PDFs ────────────────────────────────────────────
import os, shutil, glob

data_path = "/kaggle/working/rag-anything-project/donnees_rag"
os.makedirs(data_path, exist_ok=True)

# ── Option A : Copier depuis un Dataset Kaggle ──────────────────────────────
# Si vous avez créé un Dataset Kaggle avec vos PDFs, décommentez et adaptez :
# KAGGLE_DATASET = "/kaggle/input/votre-dataset-pdfs"
# for pdf in glob.glob(f"{KAGGLE_DATASET}/*.pdf"):
#     shutil.copy(pdf, data_path)

# ── Option B : Upload direct (dans Kaggle, utilisez le panneau "Add data") ──
# Vos fichiers uploadés sont dans /kaggle/input/
# Chercher tous les PDFs dans les inputs Kaggle
input_pdfs = glob.glob("/kaggle/input/**/*.pdf", recursive=True)
if input_pdfs:
    for pdf in input_pdfs:
        dest = os.path.join(data_path, os.path.basename(pdf))
        if not os.path.exists(dest):
            shutil.copy(pdf, dest)
    print(f"✅ {len(input_pdfs)} PDF(s) copiés depuis /kaggle/input/")

# ── Vérifier ────────────────────────────────────────────────────────────────
pdfs = [f for f in os.listdir(data_path) if f.endswith('.pdf')]
if pdfs:
    print(f"\n📄 {len(pdfs)} fichier(s) PDF prêts pour l'ingestion :")
    for p in pdfs:
        size_mb = os.path.getsize(os.path.join(data_path, p)) / 1024 / 1024
        print(f"  - {p} ({size_mb:.1f} MB)")
else:
    print("❌ Aucun PDF trouvé !")
    print("   → Ajoutez vos PDFs via 'Add data' dans Kaggle, puis relancez cette cellule.")

In [ ]:
# ── Cellule 5 : Lancer l'ingestion (GPU CUDA + Docling + modèles V5) ─────────
import os, shutil

os.chdir("/kaggle/working/rag-anything-project")

# Supprimer l'ancien index si existant (nouvelle config = nouvel index)
storage_path = "/kaggle/working/rag-anything-project/data/rag_anything_storage"
if os.path.exists(storage_path):
    shutil.rmtree(storage_path)
    print("🗑️  Ancien index supprimé")

# ── Configuration V5 optimisée ──────────────────────────────────────────────
# Parser
os.environ['PARSER']               = 'docling'          # Docling au lieu de MinerU
os.environ['PARSE_METHOD']         = 'auto'              # parsing multimodal (texte + tableaux)
os.environ['MINERU_DEVICE']        = 'cuda'              # GPU pour le parsing

# LLM — modèles optimisés pour le français et anti-hallucination
os.environ['OLLAMA_LLM_MODEL']     = 'mistral-nemo:12b'  # Réponses (meilleur FR)
os.environ['OLLAMA_EXTRACT_MODEL'] = 'qwen2.5:7b'        # Extraction entités (rapide)
os.environ['OLLAMA_EMBED_MODEL']   = 'nomic-embed-text'   # Embeddings

# Vision — optionnel (décommentez si vos PDFs contiennent des images)
# os.environ['OLLAMA_VISION_MODEL']  = 'llava:7b'  # + !ollama pull llava:7b

# Chemins Kaggle
os.environ['OUTPUT_DIR']           = '/kaggle/working/rag-anything-project/output'

print("🚀 Configuration V5 :")
print(f"   Parser      : {os.environ['PARSER']}")
print(f"   LLM réponse : {os.environ['OLLAMA_LLM_MODEL']}")
print(f"   LLM extract : {os.environ['OLLAMA_EXTRACT_MODEL']}")
print(f"   Embeddings  : {os.environ['OLLAMA_EMBED_MODEL']}")
print(f"   Device      : {os.environ['MINERU_DEVICE']}")
print()

# Vérifier le GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "⚠️ Pas de GPU détecté — activez le GPU dans Settings > Accelerator"

print("\n🔄 Lancement de l'ingestion...")
!python src/ingestion/rag_anything_pipeline.py

In [ ]:
# ── Cellule 6 : Vérifier le résultat de l'ingestion ─────────────────────────
import json, os

storage = "/kaggle/working/rag-anything-project/data/rag_anything_storage"

def load_json(path):
    if not os.path.exists(path):
        print(f"⚠️  Fichier absent : {os.path.basename(path)}")
        return {}
    with open(path) as f:
        return json.load(f)

docs      = load_json(f"{storage}/kv_store_doc_status.json")
chunks    = load_json(f"{storage}/kv_store_text_chunks.json")
entities  = load_json(f"{storage}/kv_store_full_entities.json")
relations = load_json(f"{storage}/kv_store_full_relations.json")

print("═" * 50)
print("📊 Statistiques du Knowledge Graph V5")
print("═" * 50)
print(f"  Documents ingérés  : {len(docs)}")
print(f"  Chunks de texte    : {len(chunks)}")
print(f"  Entités extraites  : {len(entities)}")
print(f"  Relations extraites: {len(relations)}")
print(f"  Ratio entités/doc  : {len(entities)/max(len(docs),1):.1f}")
print(f"  Ratio relations/doc: {len(relations)/max(len(docs),1):.1f}")

if not docs:
    print("\n❌ Aucun document ingéré — relancer la cellule 5")
    print("   Vérifiez que les PDFs sont dans donnees_rag/")
elif len(entities) <= len(docs):
    print("\n⚠️  Extraction d'entités insuffisante — vérifier les logs Ollama")
    print("   Le modèle qwen2.5:7b est-il bien téléchargé ?")
else:
    print("\n✅ Knowledge graph peuplé correctement")
    
    # Afficher quelques entités pour vérification
    entity_names = list(entities.keys())[:10]
    print(f"\n📋 Premières entités extraites :")
    for name in entity_names:
        print(f"  • {name}")

In [ ]:
# ── Cellule 7 : Test rapide — poser une question au RAG ──────────────────────
import os, sys, asyncio

os.chdir("/kaggle/working/rag-anything-project")
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from ingestion.rag_anything_pipeline import query as rag_query

# ── Posez votre question ici ──
QUESTION = "C'est quoi la date de Version en vigueur de CGI_Article 289?"

print(f"❓ Question : {QUESTION}\n")
print("⏳ Recherche en cours (mistral-nemo:12b)...\n")

answer = await rag_query(QUESTION, mode="hybrid")

print("═" * 60)
print("📝 Réponse :")
print("═" * 60)
print(answer)

In [ ]:
# ── Cellule 8 : Télécharger le knowledge graph ──────────────────────────────
import shutil

src = "/kaggle/working/rag-anything-project/data/rag_anything_storage"
dst = "/kaggle/working/rag_storage_v5_backup"

shutil.make_archive(dst, "zip", src)

print(f"✅ Archive créée : {dst}.zip")
print(f"   Taille : {os.path.getsize(dst + '.zip') / 1024 / 1024:.1f} MB")
print()
print("📥 Pour télécharger :")
print("   → Onglet 'Output' à droite → cliquez sur le fichier .zip")
print()
print("🏠 En local, décompressez dans : data/rag_anything_storage/")
print("   Puis lancez : python api.py")